# 第2章 2-2: Excel と CSV を読み込む

## 学習目標

このノートブックで学ぶこと：

- **`pd.read_csv()` / `pd.read_excel()`** で実際のファイルを DataFrame に読み込む
- Google Drive をマウントして、講座教材のサンプルデータを読む
- 読み込んだ後に **必ずやる「最初の3点セット」** (`head` / `shape` / `dtypes`)
- 実務でハマる **3大ポイント**: 文字コード、ヘッダー位置、日付列
- Excel の **シート指定**（`sheet_name`）

ここから「**自分の Excel** を読み込んで自動化する」ための土台が揃います。

所要時間の目安: 約25分

## ステップ1: Google Drive をマウント

オリエンテーションでやったのと同じです。Colab から **Drive のファイルにアクセスできるようにする** ための準備で、毎回必要です。

下のセルを実行すると、初回はブラウザで Drive アクセスの許可を求められます。許可してください。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ステップ2: データファイルの場所を変数にしておく

ファイルパスは何度も使うので、**冒頭で1回だけ変数にまとめておく** と後が楽です。「他人の Excel を扱うときに毎回パスを書き直す」のではなく、**ここを書き換えれば全部に反映される** 設計にします。

`BASE` は **教材ルート**（git clone した先）の絶対パスです。

In [ ]:
import pandas as pd

BASE = "/content/drive/MyDrive/street-academy/python-data-basics"
DATA = f"{BASE}/data/pandas"

# 中身を見て、ファイルがあることを確認
import os
print(os.listdir(DATA))

👆 `['employees.csv', 'sales_2026-01.xlsx']` のように表示されれば OK です。

**エラーになる場合**:
- `FileNotFoundError` → オリエンの `git clone` セルをまだ実行していない可能性。0-1 に戻って実行してください。
- パスの末尾に余計な空白 / 全角文字が混ざっていないか確認。

## CSV を読み込む — `pd.read_csv()`

CSV (Comma Separated Values) は **「値をカンマで区切ったテキストファイル」** で、ほぼどんなシステムからもダウンロードできる、データ交換の標準形式です。

### 一番素直な読み方

In [ ]:
df = pd.read_csv(f"{DATA}/employees.csv")
df

**1行で読み終わりです。** VBA で書くと数十行になる処理が、`pd.read_csv("パス")` だけで完了します。

### 読んだ後に必ずやる「最初の3点セット」

Excel から渡された CSV は、**そのまま信用してはいけません**。必ず以下の3つで「想定通りか」を確認します。

1. **`df.head()`** — 中身が崩れていないか目視
2. **`df.shape`** — 期待した行数・列数か
3. **`df.dtypes`** — 数値列がちゃんと数値型になっているか（文字列になっていないか）

In [ ]:
print("shape:", df.shape)
print()
print(df.dtypes)
print()
df.head()

👆 `salary` が `int64`（整数型）になっていることを確認してください。これが `object`（文字列）になっていると、後で `sum()` しても文字連結になってしまいます。

`hire_date` は今のところ `object`（文字列）です。日付として扱いたい場合は、後述の `parse_dates` を使います。

## Excel を読み込む — `pd.read_excel()`

`.xlsx` 形式の Excel ファイルは `pd.read_excel()` で読みます。CSV とほぼ同じ感覚です。

違いはひとつだけ — **シートが複数ある** ので、必要に応じて `sheet_name="シート名"` を指定します（省略すると一番左のシート）。

In [ ]:
sales = pd.read_excel(f"{DATA}/sales_2026-01.xlsx", sheet_name="売上")

print("shape:", sales.shape)
print()
print(sales.dtypes)
print()
sales.head()

👆 注目: `date` 列は最初から **`datetime64`（日付型）** になっています。Excel 側で日付セルだったので、pandas が自動で日付として認識してくれた、ということです。

CSV から日付を読むときは自動認識されないので、後述の `parse_dates` で明示的に指定する必要があります。

### Excel の全シートをまとめて読む

`sheet_name=None` を指定すると、**「シート名 → DataFrame」の辞書** が返ります。「全シートを順番に処理したい」ときに便利です（第4章で使います）。

In [ ]:
# 全シートを辞書で受け取る
all_sheets = pd.read_excel(f"{DATA}/sales_2026-01.xlsx", sheet_name=None)

print("シート名一覧:", list(all_sheets.keys()))

# シート名で取り出せる
all_sheets["売上"].head()

## 実務でハマる3大ポイント

ここからは、**他人の Excel/CSV を読むときに必ずぶつかる** 3つの問題と対処です。

### ① 文字コード — `UnicodeDecodeError` が出たら

Windows の Excel で「CSV (UTF-8)」を選ばずに保存すると、**Shift_JIS（cp932）** で書き出されます。これを `pd.read_csv()` で素直に読むと、文字化けや `UnicodeDecodeError` になります。

対処は **`encoding="cp932"`** を渡すこと（`shift_jis` でも可ですが、Windows 由来なら `cp932` が無難）。

```python
# UTF-8 で書かれた CSV（このノートで使ったやつ）
pd.read_csv("data.csv")                       # ← デフォルトは UTF-8

# Windows Excel から書き出した CSV
pd.read_csv("data.csv", encoding="cp932")     # ← Shift_JIS 系
```

**判別のコツ**: メモ帳で開いて文字化けしないなら UTF-8、Excel でしか開けないなら大体 cp932 です。迷ったら **両方試す** のが現実的です。

### ② ヘッダーの位置 — 上に空行や説明行があるとき

Excel のレポートは、**1行目に「2026年1月度 売上一覧」のようなタイトル**、2〜3行目に空行、4行目に列名… のように **見た目重視で作られている** ことがよくあります。

そのまま読むと「列名がタイトル行になってしまう」ので、**`skiprows`** と **`header`** で対処します。

```python
# 上から3行スキップして、4行目を列名として読む
pd.read_csv("report.csv", skiprows=3)

# 同じことを header で指定（0始まり）
pd.read_csv("report.csv", header=3)

# 列名がそもそも無い場合
pd.read_csv("raw.csv", header=None,
            names=["日付", "商品", "数量"])
```

### ③ 日付の認識 — CSV から日付を読むとき

上で読んだ Excel は日付セルだったので `datetime64` で読めましたが、**CSV から日付を読むとデフォルトでは文字列** になります。日付として扱いたい列を **`parse_dates=["列名"]`** で指定します。

やってみましょう:

In [ ]:
# 日付認識なし（hire_date は文字列）
df_no = pd.read_csv(f"{DATA}/employees.csv")
print("認識なし:")
print(df_no.dtypes)
print()

# 日付認識あり
df_yes = pd.read_csv(f"{DATA}/employees.csv", parse_dates=["hire_date"])
print("parse_dates 指定:")
print(df_yes.dtypes)

日付として認識されると、後で **「2020年以降に入社した人を抽出」** とか **「入社からの年数を計算」** が簡単にできるようになります（2-4 で扱います）。

## ちょっと実務寄り: 売上 Excel を読んで月合計を出す

実際に **読み込み → 中身確認 → 集計** の一連の流れをやってみます。

In [ ]:
# 1. 読み込み
sales = pd.read_excel(f"{DATA}/sales_2026-01.xlsx", sheet_name="売上")

# 2. 形を確認
print(f"取引件数: {len(sales)}件")
print(f"期間: {sales['date'].min()} 〜 {sales['date'].max()}")

# 3. 集計
print(f"\n月の売上合計: ¥{sales['amount'].sum():,}")
print(f"1取引あたり平均: ¥{sales['amount'].mean():,.0f}")
print(f"最大の取引: ¥{sales['amount'].max():,}")

# 4. 商品別の売上合計（次節 2-3 で深掘りする groupby）
print("\n商品別 売上合計:")
print(sales.groupby("product_name")["amount"].sum().sort_values(ascending=False))

**たったこれだけ** で「Excel から月次売上を集計」が完了します。VBA で同じ処理を書くと、ファイルを開く → ループでセルを読む → 商品ごとの集計箱を作る → 並び替え … と数十行になります。

## 演習

実務感のある問題を3つ用意しました。データは上で使った `employees.csv` と `sales_2026-01.xlsx` を使います。

### 問題1: 社員 CSV を読んで基本情報を表示

`employees.csv` を読み込んで、以下を表示してください:

- 全社員の人数（`len()` または `.shape[0]`）
- 給与の合計
- 給与の平均（小数点以下なし、3桁区切り）
- 給与が一番高い人の **行全体**

ヒント: 「給与が一番高い人」は `df[df["salary"] == df["salary"].max()]` で抽出できます（次節 2-3 で詳しく扱います）。

### 問題2: 売上 Excel を読んで日別合計を作る

`sales_2026-01.xlsx`（シート名 `売上`）を読み込んで、**日付ごとの売上合計** を出してください。

ヒント: `df.groupby("date")["amount"].sum()` で1行で書けます。

### 問題3: 入社年で抽出

`employees.csv` を **`parse_dates=["hire_date"]`** 付きで読み込んだ上で、**`hire_date.dt.year` が 2018 以降** の社員だけを表示してください。

ヒント:
```python
df = pd.read_csv(..., parse_dates=["hire_date"])
df[df["hire_date"].dt.year >= 2018]
```
`Series.dt.year` で年だけを取り出せます（日付型の Series 限定の便利機能）。

In [ ]:
import pandas as pd

BASE = "/content/drive/MyDrive/street-academy/python-data-basics"
DATA = f"{BASE}/data/pandas"

# 問題1: 社員 CSV を読んで基本情報を表示
# ここにコードを書いてください


# 問題2: 売上 Excel を読んで日別合計を作る
# ここにコードを書いてください


# 問題3: 入社年で抽出
# ここにコードを書いてください


<details>
<summary><b>▶ 解答を見る</b></summary>

### 問題1: 社員 CSV を読んで基本情報を表示

```python
df = pd.read_csv(f"{DATA}/employees.csv")

print(f"人数: {len(df)}人")
print(f"給与合計: ¥{df['salary'].sum():,}")
print(f"給与平均: ¥{df['salary'].mean():,.0f}")

# 給与が一番高い人
print()
print(df[df["salary"] == df["salary"].max()])
```

### 問題2: 売上 Excel を読んで日別合計を作る

```python
sales = pd.read_excel(f"{DATA}/sales_2026-01.xlsx", sheet_name="売上")

daily = sales.groupby("date")["amount"].sum()
print(daily)
```

### 問題3: 入社年で抽出

```python
df = pd.read_csv(f"{DATA}/employees.csv", parse_dates=["hire_date"])

recent = df[df["hire_date"].dt.year >= 2018]
print(recent)
```

**ポイント:**

- 読み込みは **`pd.read_csv()` / `pd.read_excel()` の1行で終わる**
- 読んだら **必ず `head` / `shape` / `dtypes` の3点セット** で確認するクセを付ける
- 日付列を扱う予定があるなら、読むときに **`parse_dates=[列名]`** を渡しておくと後が楽
- 「グループ別の集計」は **`groupby(列)["集計列"].sum()`** が定石（次節 2-3 で深掘り）

</details>

## よくあるエラーと対処

### `FileNotFoundError: [Errno 2] No such file or directory: '...'`

→ パスが間違っているか、Drive がマウントされていない。以下を確認:

1. 冒頭の `drive.mount(...)` セルを実行したか
2. `os.listdir(DATA)` で実際にファイルがあるか
3. パスにタイポ（半角/全角、`stree-academy` のようなスペルミス）がないか

### `UnicodeDecodeError: 'utf-8' codec can't decode byte 0x82 in position ...`

→ Shift_JIS で書かれた CSV を UTF-8 で読もうとしている。`encoding="cp932"` を付ける:

```python
pd.read_csv("data.csv", encoding="cp932")
```

### `ValueError: Worksheet named 'xxx' not found`

→ `sheet_name="xxx"` に渡したシート名が存在しない。**シート名は完全一致**（半角/全角スペースも区別）。実際のシート名を確認するには:

```python
all_sheets = pd.read_excel("file.xlsx", sheet_name=None)
print(list(all_sheets.keys()))
```

### 数値のはずの列が `object`（文字列）になっている

→ 元データに「-」「N/A」「空欄」などの非数値が混ざっている。確認 + 修復:

```python
df["金額"] = pd.to_numeric(df["金額"], errors="coerce")  # 変換失敗は NaN に
```

詳しくは 2-4 「データの加工」で扱います。

### `xlrd` のエラーが出る

→ `.xls`（古い形式）を読もうとしている。`.xlsx` に変換するか `pip install xlrd` が必要（Colab 上で `!pip install xlrd` で入れられます）。

## まとめ

このノートブックで学んだこと：

- **Drive をマウント** して、教材データのパスを `BASE` / `DATA` 変数にまとめる定石
- **`pd.read_csv("パス")`** で CSV を1行で読み込み
- **`pd.read_excel("パス", sheet_name="シート名")`** で Excel を読み込み
- **`sheet_name=None`** で全シートを辞書として一括取得
- 読んだ直後の **3点セット**: `df.head()` / `df.shape` / `df.dtypes`
- 実務ハマりどころ3つ:
  - **文字コード**: Windows Excel 由来なら `encoding="cp932"`
  - **ヘッダー位置**: 上に飾りがあるなら `skiprows=N` or `header=N`
  - **日付**: CSV からは `parse_dates=[列名]` で日付型に

次のノートブック `02-3_aggregate_filter.ipynb` では、読み込んだ DataFrame に対して **集計（`groupby`）と絞り込み（条件抽出）** を本格的に扱います。1-4 で `for + 辞書` で書いた処理が **1行** に化けるところを見ていきます。